In [2]:
import openpyxl
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from fake_useragent import UserAgent
import pypartpicker
from pathlib import Path

# ScraperAPI Proxy 設定
api_key = "66edc8a76a69bb88bf657e76121eed25"
proxy_url = f"http://proxy.scraperapi.com:8001?api_key={api_key}"


# 讀取 Excel 檔案
filename = "pcpartpicker_links_all.xlsx"
wb = openpyxl.load_workbook(filename)

# 整理所有工作表連結
all_product_links = {}

for sheet in wb.sheetnames:
    ws = wb[sheet]
    links = [row[0] for row in ws.iter_rows(min_row=2, values_only=True) if row[0]]
    all_product_links[sheet] = links


In [4]:
def scrape_pcpartpicker_sheet(sheet_name, links):
    client = pypartpicker.Client()

    # Selenium 設定
    ua = UserAgent()
    user_agent = ua.random
    options = Options()
    # options.add_argument("--headless")
    options.add_argument(f"user-agent={user_agent}")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f'--proxy-server={proxy_url}')
    driver = webdriver.Chrome(options=options)

    data_list = []
    wb_out = openpyxl.Workbook()
    ws_out = wb_out.active
    ws_out.append(["商品名稱", "商品價格", "商品規格", "商品評分", "商品連結", "買家評論"])

    try:
        for link in links:
            print(f"進入商品連結：{link}")
            driver.get(link)
            time.sleep(random.uniform(5, 8))

            try:
                name = driver.find_element(By.CLASS_NAME, "pageTitle").text

                # 價格
                try:
                    result = client.get_part(link)
                    price = result.cheapest_price.total if result.cheapest_price else "N/A"
                except Exception as e:
                    print(f"價格取得失敗：{e}")
                    price = "N/A"

                time.sleep(1)

                # 規格
                specs = {}
                spec_blocks = driver.find_elements(By.CSS_SELECTOR, "div.block.xs-hide.md-block.specs div.group.group--spec")
                for block in spec_blocks:
                    try:
                        title = block.find_element(By.CLASS_NAME, "group__title").text.strip()
                        content = block.find_element(By.CLASS_NAME, "group__content").text.strip()
                        specs[title] = content
                    except:
                        continue

                time.sleep(1)     

                # 評分
                try:
                    rating_ul = driver.find_element(By.CSS_SELECTOR, "div.actionBox__ratings ul.product--rating")
                    lis = rating_ul.find_elements(By.TAG_NAME, "li")
                    rating = lis[-1].text.strip() if lis else "N/A"
                except:
                    rating = "N/A"

                time.sleep(1)

                # 評論
                try:
                    reviews = driver.find_elements(By.CSS_SELECTOR, "div.partReviews__review div.partReviews__writeup.markdown")
                    comments = [r.text.strip() for r in reviews if r.text.strip()]
                except:
                    comments = []

                product_data = {
                    "name": name,
                    "price": price,
                    "specs": specs,
                    "rating": rating,
                    "link": link,
                    "comments": comments,
                }
                data_list.append(product_data)

                ws_out.append([
                    name,
                    price,
                    json.dumps(specs, ensure_ascii=False),
                    rating,
                    link,
                    "\n\n".join(comments)
                ])

                print(f"成功抓取：\n{product_data['name']}\n{product_data['price']}\n{product_data['specs']}\n{product_data['rating']}\n{product_data['link']}")
            
            except Exception as e:
                print(f"錯誤處理 {link}：{e}")

            time.sleep(random.uniform(5, 8))

        # 儲存
        json_out_name = f"json_data/{sheet_name}.json"
        with open(json_out_name, "w", encoding="utf-8") as f:
            json.dump(data_list, f, indent=2, ensure_ascii=False)
        print(f"{sheet_name} 已儲存為JSON。")
        
    
    finally:
        driver.quit()


In [4]:
# 查看有哪些分類可用
list(all_product_links.keys())

['speakers',
 'monitor',
 'headphones',
 'power-supply',
 'memory',
 'keyboard',
 'motherboard',
 'mouse',
 'wired-network-card',
 'cpu',
 'sound-card',
 'ups',
 'wireless-network-card',
 'fan-controller',
 'optical-drive',
 'case-fan',
 'case',
 'video-card',
 'external-hard-drive',
 'internal-hard-drive',
 'thermal-paste',
 'cpu-cooler']

In [3]:
sheet_to_scrape = 'speakers'
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/JCYmP6/kanto-tuk-260-w-20-channel-speakers-ca-tukmw
成功抓取：
Kanto TUK 260 W Speakers
849.99
{'Manufacturer': 'Kanto', 'Part #': 'CA-TUKMW', 'Configuration': '2.0', 'Total Wattage': '260 W', 'Frequency Response': '50 Hz - 20 kHz', 'Color': 'White / Black', 'Power (Front, Each)': '130 W'}
N/A
https://pcpartpicker.com/product/JCYmP6/kanto-tuk-260-w-20-channel-speakers-ca-tukmw
進入商品連結：https://pcpartpicker.com/product/cwjJ7P/creative-labs-pebble-20-44w-2ch-speakers-51mf1680aa000
成功抓取：
Creative Labs Pebble 2.0 4.4 W Speakers
23.74
{'Manufacturer': 'Creative Labs', 'Model': 'Pebble 2.0', 'Part #': '51MF1680AA000', 'Configuration': '2.0', 'Total Wattage': '4.4 W', 'Frequency Response': '100 Hz - 17 kHz', 'Color': 'Black', 'Power (Front, Each)': '2.2 W'}
(35 Ratings, 4.6 Average)
https://pcpartpicker.com/product/cwjJ7P/creative-labs-pebble-20-44w-2ch-speakers-51mf1680aa000
進入商品連結：https://pcpartpicker.com/product/PyC48d/kanto-tuk-260-w-20-channel-speakers-ca

In [8]:
sheet_to_scrape = "monitor"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/yMytt6/lg-65ep5g-b-650-3840x2160-120-hz-monitor-65ep5g-b
成功抓取：
LG 65EP5G-B 65.0" 3840 x 2160 120 Hz Monitor
9333.0
{'Manufacturer': 'LG', 'Part #': '65EP5G-B', 'Screen Size': '65"', 'Resolution': '3840 x 2160', 'Refresh Rate': '120 Hz', 'Response Time (G2G)': '0.1 ms', 'Panel Type': 'OLED', 'Aspect Ratio': '16:9', 'Color': 'Black', 'Brightness': '770 cd/m²', 'Pixel Pitch': '0.375 mm', 'Widescreen': 'Yes', 'Curved Screen': 'No', 'Frame Sync': 'None', 'Built-in Speakers': 'No', 'Viewing Angle': '178° H x 178° V', 'Inputs': '2 x HDMI'}
N/A
https://pcpartpicker.com/product/yMytt6/lg-65ep5g-b-650-3840x2160-120-hz-monitor-65ep5g-b
進入商品連結：https://pcpartpicker.com/product/DqQKHx/koorui-24e3-240-1920-x-1080-165-hz-monitor-24e3
成功抓取：
KOORUI 24E3 24.0" 1920 x 1080 165 Hz Monitor
119.99
{'Manufacturer': 'KOORUI', 'Part #': '24E3', 'Screen Size': '24"', 'Resolution': '1920 x 1080', 'Refresh Rate': '165 Hz', 'Response Time (G2G)': '1 ms', 'Panel Type': 'IPS', 

In [9]:
sheet_to_scrape = "headphones"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/fmkWGX/hifiman-susvara-headphones-susvara
成功抓取：
HiFiMAN Susvara Headphones
7999.0
{'Manufacturer': 'HiFiMAN', 'Part #': 'Susvara', 'Type': 'Circumaural', 'Frequency Response': '6 Hz - 75 kHz', 'Microphone': 'No', 'Wireless': 'No', 'Enclosure Type': 'Open', 'Color': 'Silver / Brown', 'Active Noise Cancelling': 'No', 'Connection': 'Stereo 3.5mm Audio\nStereo 6.3mm Audio', 'Channels': '2.0', 'Impedance': '60 Ω', 'Sensitivity at 1 V RMS': '83 dB'}
N/A
https://pcpartpicker.com/product/fmkWGX/hifiman-susvara-headphones-susvara
進入商品連結：https://pcpartpicker.com/product/tZL7YJ/hp-hyperx-cloud-ii-71-channel-headset-khx-hscp-rd
成功抓取：
HP HyperX Cloud II 7.1 Channel Headset
79.98
{'Manufacturer': 'HP', 'Model': 'HyperX Cloud II', 'Part #': 'KHX-HSCP-RD\n4P5M0AA', 'Type': 'Circumaural', 'Frequency Response': '15 Hz - 25 kHz', 'Microphone': 'Yes', 'Wireless': 'No', 'Enclosure Type': 'Closed', 'Color': 'Black / Red', 'Active Noise Cancelling': 'No', 'Connection':

In [5]:
sheet_to_scrape = "power-supply"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/dbCZxr/msi-mag-a750gl-pcie5-750-w-80-gold-certified-fully-modular-atx-power-supply-mag-a750gl-pcie5
成功抓取：
MSI MAG A750GL PCIE5 750 W 80+ Gold Certified Fully Modular ATX Power Supply
109.99
{'Manufacturer': 'MSI', 'Part #': 'MAG A750GL PCIE5\n306-7ZP8B11-CE0', 'Type': 'ATX', 'Efficiency Rating': '80+ Gold', 'Wattage': '750 W', 'Length': '140 mm', 'Modular': 'Full', 'Color': 'Black', 'Fanless': 'No', 'ATX 4-pin Connectors': '0', 'EPS 8-pin Connectors': '2', 'PCIe 16-pin 12VHPWR/12V-2x6 Connectors': '1', 'PCIe 12-pin Connectors': '0', 'PCIe 8-pin Connectors': '0', 'PCIe 6+2-pin Connectors': '3', 'PCIe 6-pin Connectors': '0', 'SATA Connectors': '8', 'AMP/Molex 4-pin Connectors': '4'}
(54 Ratings, 4.7 Average)
https://pcpartpicker.com/product/dbCZxr/msi-mag-a750gl-pcie5-750-w-80-gold-certified-fully-modular-atx-power-supply-mag-a750gl-pcie5
進入商品連結：https://pcpartpicker.com/product/YRJp99/corsair-rm750e-2023-750-w-80-gold-certified-fully-modular-atx-po

In [4]:
sheet_to_scrape = "memory"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/kTJp99/corsair-vengeance-rgb-32-gb-2-x-16-gb-ddr5-6000-cl36-memory-cmh32gx5m2e6000c36
成功抓取：
Corsair Vengeance RGB 32 GB (2 x 16 GB) DDR5-6000 CL36 Memory
102.99
{'Manufacturer': 'Corsair', 'Part #': 'CMH32GX5M2E6000C36', 'Speed': 'DDR5-6000', 'Form Factor': '288-pin DIMM (DDR5)', 'Modules': '2 x 16GB', 'Price / GB': '$3.218', 'Color': 'Black', 'First Word Latency': '12 ns', 'CAS Latency': '36', 'Voltage': '1.4 V', 'Timing': '36-44-44-96', 'ECC / Registered': 'Non-ECC / Unbuffered', 'Heat Spreader': 'Yes'}
(56 Ratings, 4.6 Average)
https://pcpartpicker.com/product/kTJp99/corsair-vengeance-rgb-32-gb-2-x-16-gb-ddr5-6000-cl36-memory-cmh32gx5m2e6000c36
進入商品連結：https://pcpartpicker.com/product/p6RFf7/corsair-memory-cmk16gx4m2b3200c16
成功抓取：
Corsair Vengeance LPX 16 GB (2 x 8 GB) DDR4-3200 CL16 Memory
37.99
{'Manufacturer': 'Corsair', 'Part #': 'CMK16GX4M2B3200C16', 'Speed': 'DDR4-3200', 'Form Factor': '288-pin DIMM (DDR4)', 'Modules': '2 x 8GB', 'Price /

In [11]:
sheet_to_scrape = "keyboard"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/BvL48d/hp-hyperx-alloy-core-rgb-wired-gaming-keyboard-hx-kb5me2-us
成功抓取：
HP HyperX Alloy Core RGB Wired Gaming Keyboard
39.99
{'Manufacturer': 'HP', 'Part #': 'HX-KB5ME2-US\n4P4F5AA#ABA\n4P4F5AA', 'Style': 'Gaming', 'Mechanical': 'No', 'Layout': 'English (US)', 'Backlit': 'RGB', 'Tenkeyless': 'No', 'Connection Type': 'Wired', 'Color': 'Black', 'Mouse Included': 'No'}
(22 Ratings, 4.8 Average)
https://pcpartpicker.com/product/BvL48d/hp-hyperx-alloy-core-rgb-wired-gaming-keyboard-hx-kb5me2-us
進入商品連結：https://pcpartpicker.com/product/6yQKHx/rk-royal-kludge-rk61-bluetoothwirelesswired-mini-keyboard-093348322
成功抓取：
RK Royal Kludge RK61 Bluetooth/Wireless/Wired Mini Keyboard
49.99
{'Manufacturer': 'RK Royal Kludge', 'Part #': '093348322', 'Style': 'Mini', 'Mechanical': 'Yes', 'Switch Type': 'RK Red', 'Layout': 'English (US)', 'Backlit': 'White', 'Tenkeyless': 'Yes', 'Connection Type': 'Wired\nWireless\nBluetooth Wireless', 'Normal Keys': '61', 'Color': 

In [5]:
sheet_to_scrape = "motherboard"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/szfxFT/msi-b650-gaming-plus-wifi-atx-am5-motherboard-b650-gaming-plus-wifi
成功抓取：
MSI B650 GAMING PLUS WIFI ATX AM5 Motherboard
169.99
{'Manufacturer': 'MSI', 'Part #': 'B650 GAMING PLUS WIFI\n7E26-001R\n911-7E26-001', 'Socket / CPU': 'AM5', 'Form Factor': 'ATX', 'Chipset': 'AMD B650', 'Memory Max': '192 GB', 'Memory Type': 'DDR5', 'Memory Slots': '4', 'Memory Speed': 'DDR5-4800\nDDR5-5000\nDDR5-5200\nDDR5-5400\nDDR5-5600\nDDR5-5800\nDDR5-6000\nDDR5-6200\nDDR5-6400', 'Color': 'Black', 'PCIe x16 Slots': '2', 'PCIe x1 Slots': '1', 'M.2 Slots': '2280/22110 M-key\n2260/2280 M-key', 'SATA 6.0 Gb/s Ports': '4', 'Onboard Ethernet': '1 x 2.5 Gb/s Ports (Realtek 8125BG)', 'Onboard Video': 'Depends on CPU', 'USB 2.0 Headers': '2', 'USB 3.2 Gen 1 Headers': '1', 'USB 3.2 Gen 2 Headers': '1', 'Supports ECC': 'No', 'Wireless Networking': 'Wi-Fi 6E', 'RAID Support': 'Yes', 'Uses Back-Connect Connectors': 'No'}
(45 Ratings, 4.7 Average)
https://pcpartpicker.com/p

In [12]:
sheet_to_scrape = "mouse"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/Yrw7YJ/logitech-g305-lightspeed-wirelesswired-optical-mouse-910-005280
成功抓取：
Logitech G305 LIGHTSPEED Wireless/Wired Optical Mouse
29.79
{'Manufacturer': 'Logitech', 'Part #': '910-005280\n910-005283\n910-005282', 'Tracking Method': 'Optical', 'Connection Type': 'Wired\nWireless', 'Maximum DPI': '12000', 'Hand Orientation': 'Right', 'Color': 'Black'}
(70 Ratings, 4.7 Average)
https://pcpartpicker.com/product/Yrw7YJ/logitech-g305-lightspeed-wirelesswired-optical-mouse-910-005280
進入商品連結：https://pcpartpicker.com/product/7RbwrH/logitech-g502-hero-wired-optical-mouse-910-005469
成功抓取：
Logitech G502 Hero Wired Optical Mouse
42.82
{'Manufacturer': 'Logitech', 'Part #': '910-005469\n910-005470\n910-005471', 'Tracking Method': 'Optical', 'Connection Type': 'Wired', 'Maximum DPI': '25600', 'Hand Orientation': 'Right', 'Color': 'Black'}
(201 Ratings, 4.8 Average)
https://pcpartpicker.com/product/7RbwrH/logitech-g502-hero-wired-optical-mouse-910-005469
進入商品連結

In [13]:
sheet_to_scrape = "wired-network-card"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/tqJgXL/startech-st10gpexndpi-2-x-10-gbs-ethernet-pcie-x4-network-adapter-st10gpexndpi
成功抓取：
StarTech ST10GPEXNDPI 2 x 10 Gb/s Ethernet PCIe x4 Network Adapter
438.65
{'Manufacturer': 'StarTech', 'Part #': 'ST10GPEXNDPI', 'Interface': 'PCIe x4', 'Color': 'Black / Silver'}
N/A
https://pcpartpicker.com/product/tqJgXL/startech-st10gpexndpi-2-x-10-gbs-ethernet-pcie-x4-network-adapter-st10gpexndpi
進入商品連結：https://pcpartpicker.com/product/dQmLrH/tp-link-wired-network-card-tg3468
成功抓取：
TP-Link TG-3468 Gigabit Ethernet PCIe x1 Network Adapter
14.99
{'Manufacturer': 'TP-Link', 'Part #': 'TG-3468', 'Interface': 'PCIe x1', 'Features': 'Auto-Negotiation and Auto MDI/MDIX\nSupports power down/link down power saving\nDOS/Win98SE/Me/2000/XP/Vista/Linux/Novell Netware'}
(17 Ratings, 4.9 Average)
https://pcpartpicker.com/product/dQmLrH/tp-link-wired-network-card-tg3468
進入商品連結：https://pcpartpicker.com/product/BRkwrH/intel-x540-t2-2-x-10-gbs-ethernet-pcie-x8-network-

In [3]:
sheet_to_scrape = "cpu"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/fPyH99/amd-ryzen-7-9800x3d-47-ghz-8-core-processor-100-1000001084wof
成功抓取：
AMD Ryzen 7 9800X3D 4.7 GHz 8-Core Processor
536.67
{'Manufacturer': 'AMD', 'Part #': '100-1000001084WOF\nAMD Ryzen 7 9800X3D\n100-100001084WOF', 'Series': 'AMD Ryzen 7', 'Microarchitecture': 'Zen 5', 'Core Family': 'Granite Ridge', 'Socket': 'AM5', 'Core Count': '8', 'Thread Count': '16', 'Performance Core Clock': '4.7 GHz', 'Performance Core Boost Clock': '5.2 GHz', 'L2 Cache': '8 MB', 'L3 Cache': '96 MB', 'TDP': '120 W', 'Integrated Graphics': 'Radeon', 'Maximum Supported Memory': '192 GB', 'ECC Support': 'Yes', 'Includes Cooler': 'No', 'Packaging': 'Boxed', 'Lithography': '4 nm', 'Includes CPU Cooler': 'No', 'Simultaneous Multithreading': 'Yes'}
(141 Ratings, 4.9 Average)
https://pcpartpicker.com/product/fPyH99/amd-ryzen-7-9800x3d-47-ghz-8-core-processor-100-1000001084wof
進入商品連結：https://pcpartpicker.com/product/66C48d/amd-ryzen-5-7600x-47-ghz-6-core-processor-100-10000

In [15]:
sheet_to_scrape = "sound-card"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/77VBD3/creative-labs-sound-blasterx-ae-5-plus-32-bit-384-khz-sound-card-70sb174000003
成功抓取：
Creative Labs Sound BlasterX AE-5 Plus 32-bit 384 kHz Sound Card
161.49
{'Manufacturer': 'Creative Labs', 'Model': 'Sound BlasterX AE-5 Plus', 'Part #': '70SB174000003', 'Channels': '5.1', 'Digital Audio': '32-bit', 'Signal-To-Noise Ratio': '122 dB', 'Sample Rate': '384 kHz', 'Chipset': 'Sound Core3D', 'Interface': 'PCIe x1', 'Color': 'Black'}
(15 Ratings, 4.7 Average)
https://pcpartpicker.com/product/77VBD3/creative-labs-sound-blasterx-ae-5-plus-32-bit-384-khz-sound-card-70sb174000003
進入商品連結：https://pcpartpicker.com/product/BVxfrH/creative-labs-sound-card-70sb151000000
成功抓取：
Creative Labs ZXR 24-bit 192 kHz Sound Card
498.0
{'Manufacturer': 'Creative Labs', 'Model': 'ZXR', 'Part #': '70SB151000000', 'Channels': '5.1', 'Digital Audio': '24-bit', 'Signal-To-Noise Ratio': '124 dB', 'Sample Rate': '192 kHz', 'Interface': 'PCIe x1'}
(8 Ratings, 4.6 Average)
ht

In [9]:
sheet_to_scrape = "ups"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/fzM323/apc-ups-surt20krmxlt
成功抓取：
APC SURT20KRMXLT UPS
N/A
{'Manufacturer': 'APC', 'Part #': 'SURT20KRMXLT', 'Capacity (W)': '16000 W', 'Capacity (VA)': '20000 VA', 'Rack Height': '12U', 'Backup/Run Time (Full Load)': '4.90 Minutes', 'Backup/Run Time (Half Load)': '15.30 Minutes', 'Battery Chemistry': 'Sealed Lead Acid', 'Emergency Power OFF': 'Yes', 'Form Factor': 'Tower/Rack Mountable', 'Hot Swappable': 'Yes', 'Input Voltage': '220 V AC', 'Maximum Battery Recharge Time': '2.50 Hour', 'Output Voltage': '208 V AC', 'Receptacles': '2 x NEMA L6-20R\n4 x NEMA L6-30R\n1 x Hard Wire 3-wire (2PH + G)', 'Serial Port': 'Yes'}
N/A
https://pcpartpicker.com/product/fzM323/apc-ups-surt20krmxlt
進入商品連結：https://pcpartpicker.com/product/qvVBD3/cyberpower-cp1500pfclcd-ups-cp1500pfclcd
成功抓取：
CyberPower CP1500PFCLCD UPS
239.95
{'Manufacturer': 'CyberPower', 'Part #': 'CP1500PFCLCD', 'Capacity (W)': '1000 W', 'Capacity (VA)': '1500 VA', 'Backup/Run Time (Full Load)'

In [16]:
sheet_to_scrape = "wireless-network-card"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/nRM48d/gigabyte-gc-wbax210-80211abgnacax-pcie-x1-wi-fi-adapter-gc-wbax210
成功抓取：
Gigabyte GC-WBAX210 802.11a/b/g/n/ac/ax PCIe x1 Wi-Fi Adapter
46.99
{'Manufacturer': 'Gigabyte', 'Part #': 'GC-WBAX210', 'Protocol': 'Wi-Fi 6E', 'Interface': 'PCIe x1', 'Color': 'Black / Silver'}
(10 Ratings, 4.8 Average)
https://pcpartpicker.com/product/nRM48d/gigabyte-gc-wbax210-80211abgnacax-pcie-x1-wi-fi-adapter-gc-wbax210
進入商品連結：https://pcpartpicker.com/product/8VMFf7/asus-wireless-network-card-pceac68
成功抓取：
Asus PCE-AC68 802.11a/b/g/n/ac PCIe x1 Wi-Fi Adapter
139.0
{'Manufacturer': 'Asus', 'Part #': 'PCE-AC68\n90IG00R0-BM0G00', 'Protocol': 'Wi-Fi 5', 'Interface': 'PCIe x1', 'Security': '64-bit WEP, 128-bit WEP, WPA2-PSK, WPA-PSK', 'Antenna': '3 x R SMA Antenna', 'Features': 'Data Rate: AC1750 ultimate AC performance; 450+1300Mbps\n802.11a/b/g/n/ac: downlink up to 1300Mbps, uplink up to 1300Mbps (20/40/80MHz)\nManagement: Wireless configuration\nconnection manage

In [17]:
sheet_to_scrape = "fan-controller"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/cyNYcf/arctic-case-fan-hub-fan-controller-acfan00175a
成功抓取：
ARCTIC Case Fan Hub Fan Controller
9.99
{'Manufacturer': 'ARCTIC', 'Model': 'Case Fan Hub', 'Part #': 'ACFAN00175A', 'Channels': '10', 'PWM (4-Pin)': 'Yes', 'Form Factor': 'Internal', 'Color': 'Black'}
(9 Ratings, 4.7 Average)
https://pcpartpicker.com/product/cyNYcf/arctic-case-fan-hub-fan-controller-acfan00175a
進入商品連結：https://pcpartpicker.com/product/2YMMnQ/lian-li-uni-hub-sl-controller-fan-controller-uf-in-l-connect3-wt
成功抓取：
Lian Li UNI HUB – SL Controller Fan Controller
N/A
{'Manufacturer': 'Lian Li', 'Model': 'UNI HUB – SL Controller', 'Part #': 'UF-IN-L-CONNECT3 WT\n12SL-CONT3', 'Channels': '4', 'PWM (4-Pin)': 'Yes', 'Form Factor': 'Internal', 'Color': 'Black'}
(3 Ratings, 5.0 Average)
https://pcpartpicker.com/product/2YMMnQ/lian-li-uni-hub-sl-controller-fan-controller-uf-in-l-connect3-wt
進入商品連結：https://pcpartpicker.com/product/vB8bt6/nzxt-rgb-fan-controller-2022-fan-controller-ac-

In [18]:
sheet_to_scrape = "optical-drive"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/z2dqqs/lg-optical-drive-wh14ns40
成功抓取：
LG WH14NS40 Blu-Ray/DVD/CD Writer
83.39
{'Manufacturer': 'LG', 'Part #': 'WH14NS40', 'Form Factor': '5.25"', 'Interface': 'SATA 1.5 Gb/s', 'Buffer Cache': '4 MB', 'BD-ROM Speed': '12X', 'DVD-ROM Speed': '16X', 'CD-ROM Speed': '48X', 'BD-R Speed': '14X', 'BD-R Dual-Layer Speed': '12X', 'BD-RE Speed': '2X', 'BD-RE Dual-Layer Speed': '2X', 'DVD+R Speed': '16X', 'DVD+RW Speed': '8X', 'DVD+R Dual-Layer Speed': '8X', 'DVD-R Speed': '16X', 'DVD-RW Speed': '6X', 'DVD-R Dual-Layer Speed': '8X', 'DVD-RAM Speed': '5X', 'CD-R Speed': '48X', 'CD-RW Speed': '24X'}
(168 Ratings, 4.5 Average)
https://pcpartpicker.com/product/z2dqqs/lg-optical-drive-wh14ns40
進入商品連結：https://pcpartpicker.com/product/qHdqqs/asus-optical-drive-bw16d1ht
成功抓取：
Asus BW-16D1HT Blu-Ray/DVD/CD Writer
124.08
{'Manufacturer': 'Asus', 'Part #': 'BW-16D1HT', 'Form Factor': '5.25"', 'Interface': 'SATA 3.0 Gb/s', 'BD-ROM Speed': '12X', 'DVD-ROM Speed': '16X

In [6]:
sheet_to_scrape = "case-fan"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/WNbTwP/lian-li-uni-fan-sl-infinity-613-cfm-120-mm-fans-3-pack-uf-slin120-3b
成功抓取：
Lian Li Uni Fan SL-Infinity 61.3 CFM 120 mm Fans 3-Pack
89.95
{'Manufacturer': 'Lian Li', 'Part #': 'UF-SLIN120-3B\n12SLIN3B\nG99.12SLIN3B.00', 'Size': '120 mm', 'Color': 'Black', 'Quantity': '3-Pack', 'Flow Direction': 'Standard', 'Airflow': '0 - 61.3 CFM', 'Noise Level': '0 - 29 dB', 'PWM': 'Yes', 'LED': 'Addressable RGB', 'Connector': '4-pin PWM + 3-pin 5V Addressable RGB', 'Controller': '5V Addressable RGB', 'Static Pressure': '2.66 mmH₂O'}
(61 Ratings, 4.7 Average)
https://pcpartpicker.com/product/WNbTwP/lian-li-uni-fan-sl-infinity-613-cfm-120-mm-fans-3-pack-uf-slin120-3b
進入商品連結：https://pcpartpicker.com/product/4q7G3C/noctua-nf-a12x25-pwm-chromaxblackswap-6009-cfm-120-mm-fan-nf-a12x25-pwm-chromaxblackswap
成功抓取：
Noctua NF-A12x25 PWM chromax.black.swap 60.09 CFM 120 mm Fan
37.95
{'Manufacturer': 'Noctua', 'Part #': 'NF-A12x25 PWM chromax.black.swap\nNF-A12X25 PWM

In [6]:
sheet_to_scrape = "case"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/bCYQzy/corsair-4000d-airflow-atx-mid-tower-case-cc-9011200-ww
成功抓取：
Corsair 4000D Airflow ATX Mid Tower Case
N/A
{'Manufacturer': 'Corsair', 'Part #': 'CC-9011200-WW', 'Type': 'ATX Mid Tower', 'Color': 'Black', 'Power Supply': 'None', 'Side Panel': 'Tinted Tempered Glass', 'Power Supply Shroud': 'Yes', 'Front Panel USB': 'USB 3.2 Gen 2 Type-C\nUSB 3.2 Gen 1 Type-A', 'Motherboard Form Factor': 'ATX\nEATX\nMicro ATX\nMini ITX', 'Maximum Video Card Length': '360 mm / 14.173"', 'Drive Bays': '2 x Internal 3.5"\n2 x Internal 2.5"', 'Expansion Slots': '7 x Full-Height\n2 x Full-Height via Riser', 'Dimensions': '453 mm x 230 mm x 466 mm\n17.835" x 9.055" x 18.346"', 'Volume': '48.553 L\n1.715 ft³'}
(371 Ratings, 4.7 Average)
https://pcpartpicker.com/product/bCYQzy/corsair-4000d-airflow-atx-mid-tower-case-cc-9011200-ww
進入商品連結：https://pcpartpicker.com/product/fc88TW/montech-xr-atx-mid-tower-case-xr-b
成功抓取：
Montech XR ATX Mid Tower Case
80.9
{'Manufacturer

In [4]:
sheet_to_scrape = "video-card"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/7s88TW/msi-ventus-2x-black-oc-geforce-rtx-4060-8-gb-video-card-rtx-4060-ventus-2x-black-8g-oc
成功抓取：
MSI VENTUS 2X BLACK OC GeForce RTX 4060 8 GB Video Card
342.98
{'Manufacturer': 'MSI', 'Part #': 'RTX 4060 VENTUS 2X BLACK 8G OC\nGeForce RTX 4060 VENTUS 2X BLACK 8G OC\n912-V516-004\nV516-004R\nG4060V2XB8C', 'Chipset': 'GeForce RTX 4060', 'Memory': '8 GB', 'Memory Type': 'GDDR6', 'Core Clock': '1830 MHz', 'Boost Clock': '2505 MHz', 'Interface': 'PCIe x16', 'Color': 'Black', 'Frame Sync': 'G-Sync', 'Length': '199 mm', 'TDP': '115 W', 'Case Expansion Slot Width': '2', 'Total Slot Width': '2', 'Cooling': '2 Fans', 'External Power': '1 x PCIe 8-pin', 'HDMI 2.1a Outputs': '1', 'DisplayPort 1.4a Outputs': '3'}
(10 Ratings, 4.5 Average)
https://pcpartpicker.com/product/7s88TW/msi-ventus-2x-black-oc-geforce-rtx-4060-8-gb-video-card-rtx-4060-ventus-2x-black-8g-oc
進入商品連結：https://pcpartpicker.com/product/Bvjv6h/sapphire-pulse-radeon-rx-9070-xt-16-gb-video-ca

In [19]:
sheet_to_scrape = "external-hard-drive"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/3MQKHx/apricorn-aegis-fortress-l3-20-tb-external-ssd-afl3-s20tb
成功抓取：
Apricorn Aegis Fortress L3 20 TB External SSD
12999.0
{'Manufacturer': 'Apricorn', 'Part #': 'AFL3-S20TB', 'Type': 'Portable', 'Interface': 'USB Type-A 3.2 Gen 1\nUSB Type-C 3.2 Gen 1', 'Capacity': '20000 GB', 'Price / GB': '$0.650', 'Color': 'Black', 'Cache': '8 MB', 'RPM': 'SSD'}
N/A
https://pcpartpicker.com/product/3MQKHx/apricorn-aegis-fortress-l3-20-tb-external-ssd-afl3-s20tb
進入商品連結：https://pcpartpicker.com/product/2rsV3C/western-digital-my-book-duo-44-tb-external-hard-drive-wdbfbe0440jbk-nesn
成功抓取：
Western Digital My Book Duo 44 TB External Hard Drive
1199.99
{'Manufacturer': 'Western Digital', 'Part #': 'WDBFBE0440JBK-NESN', 'Type': 'Desktop', 'Interface': 'USB Type-A 3.2 Gen 1\nUSB Type-C 3.2 Gen 1', 'Capacity': '44000 GB', 'Price / GB': '$0.027', 'Color': 'Black'}
N/A
https://pcpartpicker.com/product/2rsV3C/western-digital-my-book-duo-44-tb-external-hard-drive-wdbfbe04

In [5]:
sheet_to_scrape = "internal-hard-drive"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/34ytt6/samsung-990-pro-2-tb-m2-2280-pcie-40-x4-nvme-solid-state-drive-mz-v9p2t0bw
成功抓取：
Samsung 990 Pro 2 TB M.2-2280 PCIe 4.0 X4 NVME Solid State Drive
169.99
{'Manufacturer': 'Samsung', 'Part #': 'MZ-V9P2T0BW\nMZ-V9P2T0B/AM', 'Capacity': '2 TB', 'Price / GB': '$0.085', 'Type': 'SSD', 'Cache': '2048 MB', 'Form Factor': 'M.2-2280', 'Interface': 'M.2 PCIe 4.0 X4', 'NVME': 'Yes', 'Full Disk Write Throughput': '1479 MB/s\n(#108 of 357 tested)', 'Full Disk Write Throughput (Lowest 10 Seconds)': '1332 MB/s\n(#44 of 357 tested)', 'Random Read Throughput QD1': '55.7 MB/s\n(#61 of 357 tested)', 'Random Read Throughput QD32': '1676 MB/s\n(#17 of 357 tested)', 'Random Write Throughput QD1': '278 MB/s\n(#121 of 357 tested)', 'Random Write Throughput QD32': '2116 MB/s\n(#41 of 357 tested)', 'Sequential Read Throughput QD1': '2647 MB/s\n(#157 of 357 tested)', 'Sequential Read Throughput QD4': '7171 MB/s\n(#41 of 357 tested)', 'Sequential Write Throughput QD1'

In [7]:
sheet_to_scrape = "thermal-paste"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/6RrG3C/arctic-silver-thermal-paste-as535g
成功抓取：
Arctic Silver 5 High-Density Polysynthetic Silver 3.5 g Thermal Paste
7.95
{'Manufacturer': 'Arctic Silver', 'Model': '5 High-Density Polysynthetic Silver', 'Part #': 'AS5-3.5G\nAS-AS5-35', 'Amount': '3.5 g', 'Features': 'Three unique shapes and sizes of pure silver particles to maximize particle-to-particle contact area and thermal transfer.\nContains over 88% thermally conductive filler by weight.These thermally-enhanced ceramic particles improve the compound\'s performance and long-term stability\nEnsure the best physical contact between the heatsink and the CPU core.\nWill not separate, run, migrate, or bleed.\n3.5-gram\nAverage Particle Size: <0.49 micron <0.000020 inch\nExtended Temperature Limits: Peak: -50 Degrees C to >180 Degrees C\nLong-Term: -50 Degrees C to 130 Degrees C\nPerformance: 3 to 12 degrees centigrade lower CPU full load core temperatures than standard thermal compounds or the

In [8]:
sheet_to_scrape = "cpu-cooler"
scrape_pcpartpicker_sheet(sheet_to_scrape, all_product_links[sheet_to_scrape])

進入商品連結：https://pcpartpicker.com/product/hYxRsY/thermalright-peerless-assassin-120-se-6617-cfm-cpu-cooler-pa120-se-d3
成功抓取：
Thermalright Peerless Assassin 120 SE 66.17 CFM CPU Cooler
34.9
{'Manufacturer': 'Thermalright', 'Model': 'Peerless Assassin 120 SE', 'Part #': 'PA120 SE-D3\nPA120 SE\nPA120 SE D6-CAD\nPA120 SE 1700\nPA120 SE 1700-d6\n419043', 'Fan RPM': '1550 RPM', 'Noise Level': '25.6 dB', 'Color': 'Black / Silver', 'Height': '155 mm', 'CPU Socket': 'AM4\nAM5\nLGA1150\nLGA1151\nLGA1155\nLGA1156\nLGA1200\nLGA1700\nLGA1851', 'Water Cooled': 'No', 'Fanless': 'No'}
(110 Ratings, 4.8 Average)
https://pcpartpicker.com/product/hYxRsY/thermalright-peerless-assassin-120-se-6617-cfm-cpu-cooler-pa120-se-d3
進入商品連結：https://pcpartpicker.com/product/HyTPxr/cooler-master-hyper-212-black-edition-42-cfm-cpu-cooler-rr-212s-20pk-r1
成功抓取：
Cooler Master Hyper 212 Black Edition 42 CFM CPU Cooler
29.99
{'Manufacturer': 'Cooler Master', 'Model': 'Hyper 212 Black Edition', 'Part #': 'RR-212S-20PK-R1', 'Fa